In [66]:
from typing import Tuple
import math
import itertools

import numpy as np
import scipy as sp

In [67]:
n = 15
strike_price = 100.0
risk_free_interest_rate = 0.04
t_final = 1.0
sigma = np.array([0.30, 0.35, 0.40])
rho = np.array([[1., .5, .5],[.5, 1., .5],[.5, .5, 1.]])
s_0 = np.array([100., 100., 100.])
alpha = 2.85

In [68]:
# log grid
x_0 = np.log(s_0)
dim = len(x_0)
N = n ** dim  # physical state size (all nodes kept)
half = alpha * sigma * np.sqrt(t_final)  # per-direction half-width
x_l = x_0 - half
x_h = x_0 + half
delta_x = (x_h - x_l) / (n - 1)
x_g = [np.linspace(x_l[d], x_h[d], n) for d in range(dim)]

![image](./docs/minimum_rainbow_call_payoff.png)

In [69]:
T = sp.sparse.diags([-1.0, 0.0, 1.0], [-1, 0, 1], shape=(n, n)).toarray()
T[-1, -2:] = [-2.0, 2.0] # update due to linearity or zero-gamma boundary condition

print(f"T ({n} x {n}): \n {T}")

T (15 x 15): 
 [[ 0.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  0.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0. -1.  0.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0. -1.  0.  1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0. -1.  0.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0. -1.  0.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0. -1.  0.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0. -1.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0. -1.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0. -1.  0.  1.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. -1.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. -1.  0.  1.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. -2.  2.]]


In [70]:
S = sp.sparse.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n)).toarray()
S[-1, :] = 0.0 # update due to linearity or zero-gamma boundary condition

print(f"S ({n} x {n}): \n {S}")

S (15 x 15): 
 [[-2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 1. -2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1. -2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1. -2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  1. -2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  1. -2.  1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  1. -2.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  1. -2.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  1. -2.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  1. -2.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  1. -2.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  1. -2.  1.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  1. -2.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  1. -2.  1.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]]


In [71]:
I = sp.sparse.identity(n).toarray()

print(f"I ({n} x {n}): \n {I}")

I (15 x 15): 
 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


![image](./docs/option_price_pde.png)

In [72]:
D_1 = {} # stores first derivatives
D_2 = {} # stores second derivatives

![image](./docs/first_derivatives.png)

In [73]:
D_1[0] = (risk_free_interest_rate - 1/2 * rho[0][0] * sigma[0] ** 2) * (1 / (2 * delta_x[0])) * sp.sparse.kron(I, sp.sparse.kron(I, T))

print(f"D^(1) ({D_1[0].shape[0]} x {D_1[0].shape[1]}): \n {D_1[0].toarray()}")

D^(1) (3375 x 3375): 
 [[ 0.         -0.02046784  0.         ...  0.          0.
   0.        ]
 [ 0.02046784  0.         -0.02046784 ...  0.          0.
   0.        ]
 [ 0.          0.02046784  0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.         -0.02046784
   0.        ]
 [ 0.          0.          0.         ...  0.02046784  0.
  -0.02046784]
 [ 0.          0.          0.         ...  0.          0.04093567
  -0.04093567]]


In [74]:
D_1[1] = (risk_free_interest_rate - 1/2 * rho[1][1] * sigma[1] ** 2) * (1 / (2 * delta_x[1])) * sp.sparse.kron(I, sp.sparse.kron(T, I))

print(f"D^(2) ({D_1[1].shape[0]} x {D_1[1].shape[1]}): \n {D_1[1].toarray()}")

D^(2) (3375 x 3375): 
 [[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ... -0.14912281  0.
   0.        ]
 [ 0.          0.          0.         ...  0.         -0.14912281
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
  -0.14912281]]


In [75]:
D_1[2] = (risk_free_interest_rate - 1/2 * rho[2][2] * sigma[2] ** 2) * (1 / (2 * delta_x[2])) * sp.sparse.kron(T, sp.sparse.kron(I, I))

print(f"D^(3) ({D_1[2].shape[0]} x {D_1[2].shape[1]}): \n {D_1[2].toarray()}")

D^(3) (3375 x 3375): 
 [[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ... -0.24561404  0.
   0.        ]
 [ 0.          0.          0.         ...  0.         -0.24561404
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
  -0.24561404]]


![image](./docs/second_derivatives.png)

In [76]:
D_2[(0, 0)] = 1/2 * (sigma[0] ** 2) * (1 / delta_x[0] ** 2) * sp.sparse.kron(I, sp.sparse.kron(I, S))

print(f"D^(11) ({D_2[(0, 0)].shape[0]} x {D_2[(0, 0)].shape[1]}): \n {D_2[(0, 0)].toarray()}")

D^(11) (3375 x 3375): 
 [[-6.03262542  3.01631271  0.         ...  0.          0.
   0.        ]
 [ 3.01631271 -6.03262542  3.01631271 ...  0.          0.
   0.        ]
 [ 0.          3.01631271 -6.03262542 ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ... -6.03262542  3.01631271
   0.        ]
 [ 0.          0.          0.         ...  3.01631271 -6.03262542
   3.01631271]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


In [77]:
D_2[(1, 1)] = 1/2 * (sigma[1] ** 2) * (1 / delta_x[1] ** 2) * sp.sparse.kron(I, sp.sparse.kron(S, I))

print(f"D^(22) ({D_2[(1, 1)].shape[0]} x {D_2[(1, 1)].shape[1]}): \n {D_2[(1, 1)].toarray()}")

D^(22) (3375 x 3375): 
 [[-6.03262542  0.          0.         ...  0.          0.
   0.        ]
 [ 0.         -6.03262542  0.         ...  0.          0.
   0.        ]
 [ 0.          0.         -6.03262542 ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


In [78]:
D_2[(2, 2)] = 1/2 * (sigma[2] ** 2) * (1 / delta_x[2] ** 2) * sp.sparse.kron(S, sp.sparse.kron(I, I))

print(f"D^(33) ({D_2[(2, 2)].shape[0]} x {D_2[(2, 2)].shape[1]}): \n {D_2[(2, 2)].toarray()}")

D^(33) (3375 x 3375): 
 [[-6.03262542  0.          0.         ...  0.          0.
   0.        ]
 [ 0.         -6.03262542  0.         ...  0.          0.
   0.        ]
 [ 0.          0.         -6.03262542 ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


![image](./docs/mixed_derivatives.png)

In [79]:
D_2[(0, 1)] = 1/2 * (rho[0][1] * sigma[0] * sigma[1]) * (1 / (2 * delta_x[0])) * (1 / (2 * delta_x[1])) * sp.sparse.kron(I, sp.sparse.kron(T, T))

print(f"D^(12) ({D_2[(0, 1)].shape[0]} x {D_2[(0, 1)].shape[1]}): \n {D_2[(0, 1)].toarray()}")

D^(12) (3375 x 3375): 
 [[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.          0.75407818
   0.        ]
 [ 0.          0.          0.         ... -0.75407818  0.
   0.75407818]
 [ 0.          0.          0.         ...  0.         -1.50815636
   1.50815636]]


In [80]:
D_2[(0, 2)] = 1/2 * (rho[0][2] * sigma[0] * sigma[2]) * (1 / (2 * delta_x[0])) * (1 / (2 * delta_x[2])) * sp.sparse.kron(T, sp.sparse.kron(I, T))

print(f"D^(13) ({D_2[(0, 2)].shape[0]} x {D_2[(0, 2)].shape[1]}): \n {D_2[(0, 2)].toarray()}")

D^(13) (3375 x 3375): 
 [[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.          0.75407818
   0.        ]
 [ 0.          0.          0.         ... -0.75407818  0.
   0.75407818]
 [ 0.          0.          0.         ...  0.         -1.50815636
   1.50815636]]


In [81]:
D_2[(1, 2)] = 1/2 * (rho[1][2] * sigma[1] * sigma[2]) * (1 / (2 * delta_x[1])) * (1 / (2 * delta_x[2])) * sp.sparse.kron(T, sp.sparse.kron(T, I))

print(f"D^(23) ({D_2[(1, 2)].shape[0]} x {D_2[(1, 2)].shape[1]}): \n {D_2[(1, 2)].toarray()}")

D^(23) (3375 x 3375): 
 [[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.50815636 0.         0.        ]
 [0.         0.         0.         ... 0.         1.50815636 0.        ]
 [0.         0.         0.         ... 0.         0.         1.50815636]]


![image](./docs/discretized_matrix.png)

In [82]:
A = sp.sparse.csr_matrix((N, N))

pairs = list(itertools.combinations_with_replacement(list(range(dim)), 2))
for (i, j) in pairs:
  if i == j:
    A += D_1[i]
    A += D_2[(i, i)]
  else:
    A += 2 * D_2[(i, j)]

A -= risk_free_interest_rate * sp.sparse.identity(N)

print(f"A ({A.shape[0]} x {A.shape[1]}): \n {A.toarray()}")

A (3375 x 3375): 
 [[-18.13787627   2.99584488   0.         ...   0.           0.
    0.        ]
 [  3.03678055 -18.13787627   2.99584488 ...   0.           0.
    0.        ]
 [  0.           3.03678055 -18.13787627 ...   0.           0.
    0.        ]
 ...
 [  0.           0.           0.         ...  -3.45104955   6.01215759
    0.        ]
 [  0.           0.           0.         ...   0.02046784  -3.45104955
    6.01215759]
 [  0.           0.           0.         ...   0.          -5.99168975
    8.57326562]]


![image](./docs/sequential_arnoldi.png)

In [83]:
def sequential_arnoldi(
    A: np.ndarray,
    v_0: np.ndarray,
    m: np.int64,
    tolerance: np.float64 = 1e-6,
    ) -> Tuple[np.ndarray, np.ndarray]:
    r"""
    Arnoldi's procedure is an algorithm for building an orthogonal
      basis of the Krylov subspace \mathcal{K}_m(A, v_0)
      and the corresponding upper Hessenberg matrix.

    Args:
      A: Square input matrix (n x n)
      v_0: Initial vector (n x 1), must be non-zero
      m: Dimension of subspace \mathcal{K}
      tolerance: double precision comparison tolerance

    Returns:
      Tuple of matrix V, H. Where V is n x m matrix whose column-vectors form
        a basis of \mathcal{K}, and H is a (m + 1) x m Hessenberg matrix.
    """
    # Checks
    n = A.shape[0]
    if n != A.shape[1]:
        raise ValueError("Matrix A must be square.")
    if v_0.shape[0] != n:
        raise ValueError("v0 size must match A dimensions.")
    if m <= 0 or m > n:
        raise ValueError("m must satisfy 0 < m <= n.")

    # Algorithm
    V = np.zeros((n, m)) # n x m matrix whose column-vectors form a basis of \mathcal{K}
    H = np.zeros((m + 1, m)) # (m + 1) x m Hessenberg matrix

    V[:, 0] = v_0.flatten() / np.linalg.norm(v_0)

    for j in range(m):
        w_j = A @ V[:, j] # column vector of basis \mathcal{L}

        for i in range(j + 1):
            H[i, j] = np.dot(w_j, V[:, i])
            w_j -= H[i, j] * V[:, i]

        H[j + 1, j] =  np.linalg.norm(w_j)

        if H[j + 1, j] <= tolerance:
            print(f"[INFO] Arnoldi breakdown at j={j}: h_{{j+1,j}} approx 0.0")
            break

        if j + 1 < m:
            V[:, j + 1] = w_j / H[j + 1, j]

    return V, H

In [84]:
# Temporal Parameters
temporal_steps = 100
delta_t = t_final / temporal_steps
t = np.linspace(0., t_final, temporal_steps + 1)

payoff = np.zeros(N)
for i1 in range(n):
    for i2 in range(n):
        for i3 in range(n):
            s = np.exp([x_g[0][i1], x_g[1][i2], x_g[2][i3]])  # node's s-coordinates
            gid = i3*n*n + i2*n + i1                          # x1 fastest
            payoff[gid] = max(min([s[0], s[1], s[2]]) - strike_price, 0.0)

U = np.zeros((temporal_steps + 1, N))
U[0, :] = payoff

![image](./docs/exponential_integrator_algo.png)

In [85]:
t_curr = 0.0
tolerance = 1e-8
for time_step in range(temporal_steps):
    h = min(delta_t, t_final - t_curr)

    v = U[time_step, :].reshape(N, 1)  # no augmentation - v is just u_k

    m = 1
    err = np.inf
    while m <= N: # cap is N now, not N + p
        Q, H = sequential_arnoldi(A, v, m, tolerance)
        h_ij = H[-1, -1]
        H = H[:-1, :]
        beta = np.linalg.norm(v)
        f = sp.linalg.expm(h * H)[:, 0]
        w = (beta * Q) @ f
        err = beta * abs(h_ij) * abs(f[-1])
        if err < tolerance:
            break
        m += 1

    U[time_step + 1, :] = w # all N entries - nothing to slice off
    t_curr += h

In [86]:
# U[-1] holds the option value at every grid node, but the price we actually want is at the spot
# and the spot might not sit exactly on a node. So we have to find the node closest to it
# x_g is the array of node coordinates in direction d, living in log-space
# np.log(s_0) puts the spot into that same log-space
# The next line returns the index of the smallest distance
spot = [np.argmin(np.abs(x_g[d] - np.log(s_0[d]))) for d in range(dim)]
gid  = spot[2]*n*n + spot[1]*n + spot[0]
computed_result = U[-1, gid]
print(f"rainbow price at spot: {computed_result:.4f}")
expected_result = 4.4450
print(f"rainbow price reference: {expected_result:.4f}")
true_error = np.linalg.norm(computed_result - expected_result)
print(f"True Error (||x_m - x||) = {true_error}")

rainbow price at spot: 4.0851
rainbow price reference: 4.4450
True Error (||x_m - x||) = 0.359878365927095
